In [1]:
from datetime import datetime
from vnpy.trader.constant import Interval
from vnpy_portfoliostrategy.backtesting import BacktestingEngine

import importlib
from vnpy_portfoliostrategy.my_strategies import multi_martingale

importlib.reload(multi_martingale)
from vnpy_portfoliostrategy.my_strategies.multi_martingale import MultiAssetMartingaleStrategy

SYMBOLS = [
    # ========== 跨资产类别（核心低相关性组合）==========
    "518880.SSE",      # 黄金ETF - 与A股相关性约-0.2，避险资产
    "511010.SSE",      # 国债ETF - 与股票负相关，股债跷跷板[citation:1]
    "513100.SSE",      # 纳指ETF - 美股科技，与A股轮动[citation:1]
    
    # ========== A股价值型（低波动、防御属性）==========
    "563020.SSE",      # 红利低波ETF - 高股息+低波动，防御核心[citation:7]
    "512390.SSE",      # MSCI低波ETF - MSCI低波动因子[citation:8]
    "159201.SZSE",     # 自由现金流ETF - 财务稳健，抗跌性强[citation:3]
    
    # ========== A股成长型（高弹性、独立行情）==========
    "159259.SZSE",     # 成长ETF - 国证成长100，弹性高[citation:2]
    "512100.SSE",      # 1000ETF - 中小盘，与大盘相关性低[citation:1]
    
    # ========== 商品/周期类（与股债低相关）==========
    "159981.SZSE",     # 豆粕ETF - 农产品，与宏观周期低相关[citation:3]
    "515220.SSE",      # 煤炭ETF - 周期资源，独立行情[citation:6]
]

def get_config(symbols):
    config = {"rates": {}, "slippages": {}, "sizes": {}, "priceticks": {}}
    for s in symbols:
        config["rates"][s] = 3 / 10000
        config["slippages"][s] = 0.001
        config["sizes"][s] = 1
        config["priceticks"][s] = 0.001
    return config

engine = BacktestingEngine()
config = get_config(SYMBOLS)

engine.set_parameters(
    vt_symbols=SYMBOLS,
    interval=Interval.DAILY,
    start=datetime(2015, 1, 1),
    end=datetime(2026, 12, 31),
    rates=config["rates"],
    slippages=config["slippages"],
    sizes=config["sizes"],
    priceticks=config["priceticks"],
    capital=1_000_000,  # 修正：100万
)

# 统一配置（策略内部会通过ATR自动适应不同波动率）
setting = {
    "initial_volume": 5000,
    "multiplier": 1.3,
    "add_pct": 0.03,
    "max_add_count": 15,
    "cooldown_days": 1,
    
    # 多级止盈
    "use_multi_target": True,
    "target1_pct": 0.07,   # 2%平仓30%
    "target2_pct": 0.08,   # 5%再平仓30%
    "target3_pct": 0.09,   # 10%全部平仓
    "target1_ratio": 0.3,
    "target2_ratio": 0.3,
    "target3_ratio": 0.4,
}

engine.add_strategy(MultiAssetMartingaleStrategy, setting)
engine.load_data()
engine.run_backtesting()
df = engine.calculate_result()
engine.calculate_statistics()
engine.show_chart()

2026-06-20 12:46:38.148192	Start loading historical data
2026-06-20 12:46:39.509143	518880.SSE Historical data loading completed, data count: 2769
2026-06-20 12:46:39.631324	511010.SSE Historical data loading completed, data count: 2769
2026-06-20 12:46:39.720034	513100.SSE Historical data loading completed, data count: 2768
2026-06-20 12:46:39.750646	563020.SSE Historical data loading completed, data count: 592
2026-06-20 12:46:40.041499	512390.SSE Historical data loading completed, data count: 1808
2026-06-20 12:46:40.059312	159201.SZSE Historical data loading completed, data count: 304
2026-06-20 12:46:40.067613	159259.SZSE Historical data loading completed, data count: 179
2026-06-20 12:46:40.165713	512100.SSE Historical data loading completed, data count: 2321
2026-06-20 12:46:40.237349	159981.SZSE Historical data loading completed, data count: 1539
2026-06-20 12:46:40.300679	515220.SSE Historical data loading completed, data count: 1514
2026-06-20 12:46:40.301270	All historical d

In [2]:
print("\n" + "="*30 + " 策略日志输出 " + "="*30)
for log in engine.logs:
    print(log)  # 每条日志会包含时间和内容


============================== 策略日志输出 ==============================
1970-01-01 00:00:00	============================================================
1970-01-01 00:00:00	多资产马丁策略初始化开始（多级止盈版）
1970-01-01 00:00:00	资产数量: 10
1970-01-01 00:00:00	资产列表: ['518880.SSE', '511010.SSE', '513100.SSE', '563020.SSE', '512390.SSE', '159201.SZSE', '159259.SZSE', '512100.SSE', '159981.SZSE', '515220.SSE']
1970-01-01 00:00:00	初始开仓手数: 5000
1970-01-01 00:00:00	加仓倍数: 1.3
1970-01-01 00:00:00	加仓间距: 3.0%
1970-01-01 00:00:00	最大加仓次数: 15
1970-01-01 00:00:00	冷却天数: 1
1970-01-01 00:00:00	多级止盈: 已启用
1970-01-01 00:00:00	  目标1: 7.000000000000001% 平仓30.0%
1970-01-01 00:00:00	  目标2: 8.0% 平仓30.0%
1970-01-01 00:00:00	  目标3: 9.0% 全部平仓
1970-01-01 00:00:00	策略初始化完成
1970-01-01 00:00:00	============================================================
2015-01-05 00:00:00+08:00	
2015-01-05 00:00:00+08:00	on_bars第1次调用 - 日期:2015-01-05
2015-01-05 00:00:00+08:00	收到3个标的
2015-01-05 00:00:00+08:00	==============================================

In [3]:
if engine.trades:
    for trade in engine.trades.values():
        # 只打印 TradeData 实际有的字段
        print(f"时间: {trade.datetime} | 标的: {trade.symbol} | "
              f"方向: {trade.direction.value} | 开平: {trade.offset.value} | "
              f"价格: {trade.price:.3f} | 数量: {trade.volume}")
else:
    print("无成交记录")

时间: 2015-01-19 00:00:00+08:00 | 标的: 511010 | 方向: Long | 开平: Open | 价格: 106.462 | 数量: 5000
时间: 2015-01-19 00:00:00+08:00 | 标的: 513100 | 方向: Long | 开平: Open | 价格: 1.351 | 数量: 5000
时间: 2015-01-20 00:00:00+08:00 | 标的: 518880 | 方向: Long | 开平: Open | 价格: 2.570 | 数量: 5000
时间: 2015-01-23 00:00:00+08:00 | 标的: 518880 | 方向: Short | 开平: Close | 价格: 2.597 | 数量: 1500
时间: 2015-03-06 00:00:00+08:00 | 标的: 513100 | 方向: Short | 开平: Close | 价格: 1.462 | 数量: 1500
时间: 2015-03-20 00:00:00+08:00 | 标的: 518880 | 方向: Long | 开平: Open | 价格: 2.365 | 数量: 6500
时间: 2015-04-29 00:00:00+08:00 | 标的: 513100 | 方向: Short | 开平: Close | 价格: 1.466 | 数量: 1050
时间: 2015-07-09 00:00:00+08:00 | 标的: 518880 | 方向: Long | 开平: Open | 价格: 2.236 | 数量: 8450
时间: 2015-07-21 00:00:00+08:00 | 标的: 513100 | 方向: Short | 开平: Close | 价格: 1.523 | 数量: 2450
时间: 2015-07-22 00:00:00+08:00 | 标的: 513100 | 方向: Long | 开平: Open | 价格: 1.512 | 数量: 5000
时间: 2015-07-28 00:00:00+08:00 | 标的: 513100 | 方向: Long | 开平: Open | 价格: 1.476 | 数量: 6500
时间: 2015-08-04 00:00:0

In [4]:
from datetime import datetime
import pandas as pd
import os
from vnpy.trader.constant import Interval
from vnpy_portfoliostrategy.backtesting import BacktestingEngine

import importlib
from vnpy_portfoliostrategy.my_strategies import multi_martingale

# 重新加载模块
importlib.reload(multi_martingale)

# 重新导入类
from vnpy_portfoliostrategy.my_strategies.multi_martingale import MultiAssetMartingaleStrategy

# 标的列表
SYMBOLS = [
    # ========== 跨资产类别（核心低相关性组合）==========
    "518880.SSE",      # 黄金ETF
    "511010.SSE",      # 国债ETF
    "513100.SSE",      # 纳指ETF
    
    # ========== A股价值型（低波动、防御属性）==========
    "563020.SSE",      # 红利低波ETF
    "512390.SSE",      # MSCI低波ETF
    "159201.SZSE",     # 自由现金流ETF
    
    # ========== A股成长型（高弹性、独立行情）==========
    "159259.SZSE",     # 成长ETF
    "512100.SSE",      # 1000ETF
    
    # ========== 商品/周期类（与股债低相关）==========
    "159981.SZSE",     # 豆粕ETF
    "515220.SSE",      # 煤炭ETF
]

# 自动生成配置
def get_config(symbols):
    config = {
        "rates": {},
        "slippages": {},
        "sizes": {},
        "priceticks": {}
    }
    
    for symbol in symbols:
        config["rates"][symbol] = 3/10000      # 万三手续费
        config["slippages"][symbol] = 0.001    # 千分之一滑点
        config["sizes"][symbol] = 1            # ETF一份就是1股
        config["priceticks"][symbol] = 0.001   # 最小变动0.001元
    
    return config

# 创建回测引擎
engine = BacktestingEngine()

# 获取配置
config = get_config(SYMBOLS)

# 设置参数
engine.set_parameters(
    vt_symbols=SYMBOLS,
    interval=Interval.DAILY,
    start=datetime(2015, 1, 1),
    end=datetime(2026, 12, 31),
    rates=config["rates"],
    slippages=config["slippages"],
    sizes=config["sizes"],
    priceticks=config["priceticks"],
    capital=1_000_000,
)

# 策略参数
setting = {
    "initial_volume": 5000,
    "multiplier": 1.3,
    "add_pct": 0.03,
    "max_add_count": 15,
    "cooldown_days": 1,
    
    # 多级止盈
    "use_multi_target": True,
    "target1_pct": 0.07,   # 7%平仓30%
    "target2_pct": 0.08,   # 8%再平仓30%
    "target3_pct": 0.09,   # 9%全部平仓
    "target1_ratio": 0.3,
    "target2_ratio": 0.3,
    "target3_ratio": 0.4,
}

engine.add_strategy(MultiAssetMartingaleStrategy, setting)
engine.load_data()
engine.run_backtesting()

# 获取每日数据 - 先计算统计，再获取DataFrame
engine.calculate_result()
daily_df = engine.calculate_result()

# 如果get_daily_result不存在，手动构建DataFrame
if daily_df is None or len(daily_df) == 0:
    # 从engine中获取数据
    if hasattr(engine, 'daily_pnl'):
        daily_df = pd.DataFrame({
            'datetime': pd.Series(engine.daily_dt, dtype='datetime64'),
            'net_pnl': engine.daily_pnl,
            'balance': engine.daily_balance,
        })
        daily_df.set_index('datetime', inplace=True)
    else:
        # 使用calculate_result返回的数据
        daily_df = engine.calculate_result()

# 确保索引是日期格式
if not isinstance(daily_df.index, pd.DatetimeIndex):
    daily_df.index = pd.to_datetime(daily_df.index)
daily_df = daily_df.sort_index()

# 计算净值（使用balance列）
if 'balance' in daily_df.columns:
    daily_df['nav'] = daily_df['balance']
else:
    # 如果没有balance，从net_pnl计算
    daily_df['nav'] = 1_000_000 + daily_df['net_pnl'].cumsum()

# 找到策略真正开始交易的日期（第一次有成交或净值开始变化）
strategy_start_date = None

# 查找第一次有成交的日期
if 'trade_count' in daily_df.columns:
    first_trade_df = daily_df[daily_df['trade_count'] > 0]
    if len(first_trade_df) > 0:
        strategy_start_date = first_trade_df.index[0]

# 如果没有成交，查找净值第一次变化的日期
if strategy_start_date is None:
    initial_nav = daily_df['nav'].iloc[0]
    nav_changed_df = daily_df[abs(daily_df['nav'] - initial_nav) > 1]
    if len(nav_changed_df) > 0:
        strategy_start_date = nav_changed_df.index[0]

print(f"\n策略预热完成，正式开始交易日期: {strategy_start_date.strftime('%Y-%m-%d')}")

# 从策略真正开始日期开始，过滤掉预热期
daily_df = daily_df[daily_df.index >= strategy_start_date]

# ========== 按年统计 ==========
print("\n" + "="*80)
print("📊 按年统计结果")
print("="*80)

# 添加年份列
daily_df['year'] = daily_df.index.year

annual_results = []

for year, group in daily_df.groupby('year'):
    if len(group) == 0:
        continue
    
    # 获取该年净值序列
    nav = group['nav']
    start_nav = nav.iloc[0]
    end_nav = nav.iloc[-1]
    
    # 起始日期和结束日期
    start_date = group.index[0].strftime('%Y-%m-%d')
    end_date = group.index[-1].strftime('%Y-%m-%d')
    
    # 总收益率
    total_return = (end_nav - start_nav) / start_nav
    
    # 年化收益（基于实际交易日数）
    trading_days = len(group)
    if trading_days > 0 and total_return > -1:
        annual_return = (1 + total_return) ** (252 / trading_days) - 1
    else:
        annual_return = 0
    
    # 最大回撤（使用全局累计最高点）
    cummax = nav.cummax()
    drawdown = (nav - cummax) / cummax
    max_drawdown = drawdown.min()
    
    # 夏普比率
    daily_returns = nav.pct_change().dropna()
    if len(daily_returns) > 0 and daily_returns.std() > 0:
        sharpe = (daily_returns.mean() * 252) / (daily_returns.std() * (252 ** 0.5))
    else:
        sharpe = 0
    
    # 成交统计
    total_trades = group['trade_count'].sum() if 'trade_count' in group.columns else 0
    
    annual_results.append({
        '年份': year,
        '起始日期': start_date,
        '结束日期': end_date,
        '年化收益': annual_return,
        '总收益': total_return,
        '最大回撤': max_drawdown,
        '夏普比率': sharpe,
        '起始净值': start_nav,
        '结束净值': end_nav,
        '交易日数': trading_days,
        '成交笔数': total_trades,
    })

# 创建显示用的DataFrame
display_df = pd.DataFrame([{
    '年份': r['年份'],
    '起始日期': r['起始日期'],
    '结束日期': r['结束日期'],
    '年化收益': f"{r['年化收益']:.2%}",
    '总收益': f"{r['总收益']:.2%}",
    '最大回撤': f"{r['最大回撤']:.2%}",
    '夏普比率': f"{r['夏普比率']:.2f}",
    '起始净值': f"{r['起始净值']:,.0f}",
    '结束净值': f"{r['结束净值']:,.0f}",
    '交易日数': r['交易日数'],
    '成交笔数': r['成交笔数']
} for r in annual_results])

print(display_df.to_string(index=False))

# ========== 整体统计 ==========
print(f"\n{'='*80}")
print("📈 整体统计")
print(f"{'='*80}")

if len(annual_results) > 0:
    # 整体收益（从策略开始到结束）
    total_nav_start = annual_results[0]['起始净值']
    total_nav_end = annual_results[-1]['结束净值']
    total_return_all = (total_nav_end - total_nav_start) / total_nav_start
    
    # 整体年化收益
    total_days = sum(r['交易日数'] for r in annual_results)
    if total_days > 0:
        total_annual_return = (1 + total_return_all) ** (252 / total_days) - 1
    else:
        total_annual_return = 0
    
    # 最大回撤（所有年份中最大的）
    max_drawdown_all = min(r['最大回撤'] for r in annual_results)
    
    # 平均夏普
    avg_sharpe = sum(r['夏普比率'] for r in annual_results) / len(annual_results)
    
    # 正收益年份
    positive_years = [r for r in annual_results if r['总收益'] > 0]
    
    print(f"策略开始日期: {annual_results[0]['起始日期']}")
    print(f"策略结束日期: {annual_results[-1]['结束日期']}")
    print(f"整体总收益: {total_return_all:.2%}")
    print(f"整体年化收益: {total_annual_return:.2%}")
    print(f"最大回撤: {max_drawdown_all:.2%}")
    print(f"平均夏普比率: {avg_sharpe:.2f}")
    print(f"正收益年份: {len(positive_years)}/{len(annual_results)} ({len(positive_years)/len(annual_results):.0%})")
    print(f"总成交笔数: {sum(r['成交笔数'] for r in annual_results)}")

# 保存到CSV
desktop = os.path.join(os.path.expanduser('~'), 'Desktop')
file_path = os.path.join(desktop, 'MultiAssetMartingale_backtest_results.csv')

save_df = pd.DataFrame([{
    '年份': r['年份'],
    '起始日期': r['起始日期'],
    '结束日期': r['结束日期'],
    '年化收益_%': f"{r['年化收益']:.2%}",
    '总收益_%': f"{r['总收益']:.2%}",
    '最大回撤_%': f"{r['最大回撤']:.2%}",
    '夏普比率': f"{r['夏普比率']:.2f}",
    '起始净值': r['起始净值'],
    '结束净值': r['结束净值'],
    '交易日数': r['交易日数'],
    '成交笔数': r['成交笔数']
} for r in annual_results])

save_df.to_csv(file_path, index=False, encoding='utf-8-sig')
print(f"\n✅ 结果已保存到: {file_path}")

# 显示图表
engine.show_chart()

2026-06-20 12:46:41.378739	Start loading historical data
2026-06-20 12:46:41.398138	518880.SSE Historical data loading completed, data count: 2769
2026-06-20 12:46:41.407341	511010.SSE Historical data loading completed, data count: 2769
2026-06-20 12:46:41.415385	513100.SSE Historical data loading completed, data count: 2768
2026-06-20 12:46:41.417320	563020.SSE Historical data loading completed, data count: 592
2026-06-20 12:46:41.421277	512390.SSE Historical data loading completed, data count: 1808
2026-06-20 12:46:41.421783	159201.SZSE Historical data loading completed, data count: 304
2026-06-20 12:46:41.421935	159259.SZSE Historical data loading completed, data count: 179
2026-06-20 12:46:41.423833	512100.SSE Historical data loading completed, data count: 2321
2026-06-20 12:46:41.426514	159981.SZSE Historical data loading completed, data count: 1539
2026-06-20 12:46:41.428143	515220.SSE Historical data loading completed, data count: 1514
2026-06-20 12:46:41.428176	All historical d

KeyError: 'balance'

In [ ]:
import pandas as pd

# 创建包含balance的DataFrame
balance_df = pd.DataFrame({
    'date': df.index,
    'balance': df['balance'].values
})

# 导出为xlsx
balance_df.to_excel('strategy_balance.xlsx', index=False)
print("✅ 已导出 strategy_balance.xlsx")
print(f"数据量: {len(balance_df)} 行")
print(f"初始资金: {balance_df['balance'].iloc[0]:.2f}")
print(f"最终资金: {balance_df['balance'].iloc[-1]:.2f}")

✅ 已导出 strategy_balance.xlsx
数据量: 2760 行
初始资金: 1000000.00
最终资金: 1967086.41


In [ ]:
import pandas as pd

initial_capital = 100_0000  # 100万

# 1. 读 close_price
price_df = pd.read_excel("close_price.xlsx")

# 假设列是 close_price + date（或 index 是 date）
if "date" in price_df.columns:
    price_df["date"] = pd.to_datetime(price_df["date"])
    price_df = price_df.set_index("date")
else:
    price_df.index = pd.to_datetime(price_df.index)

# 2. 确保 df 也有 datetime index
df.index = pd.to_datetime(df.index)

# 3. 对齐时间（关键步骤）
merged = pd.DataFrame(index=df.index).join([
    df["balance"],
    price_df["close_price"]
], how="inner")

# 4. 计算 buy & hold NAV
merged["buy_hold"] = (
    merged["close_price"] / merged["close_price"].iloc[0]
) * initial_capital

# 5. 导出 Excel
merged.to_excel("backtest_result.xlsx")